<div class="alert alert-info">
    
➡️ Before you start, make sure that you are familiar with the **[study guide](https://liu-nlp.ai/text-mining/logistics/)**, in particular the rules around **cheating and plagiarism**.

➡️ Make sure you fill in **all** cells that say **`YOUR CODE HERE`** or **YOUR ANSWER HERE**.  You normally shouldn't need to modify any of the other cells.

➡️ Make sure that you save the notebook **with the outputs from all cells visible,** as if you had run the notebook from top to bottom.

</div>

# L5: Text Summarization

<div class="alert alert-warning">

⚠️ This lab **works best with a GPU with 10GB+ VRAM.**  Remember that you can access the GPU-equipped lab rooms in B-huset – Åsgard, Egypten, Olympen – to work on this lab, preferably during the times we have booked one or more of these rooms for the course.

</div>

In this lab, you will use large language models (LLMs) to generate summaries of (short) news articles.  In the first part, you will use dense document embeddings to perform extractive text summarization, while in the second part, you will use a decoder LLM to perform abstractive text summarization.

In [19]:
from liu_tm import check_solutions

## Dataset

In this lab, we will look specifically at the task of text summarization. For this, we are using a subset of the CNN/DailyMail&nbsp;3.0.0 data set, a popular dataset for summarization composed of news articles.

In [20]:
import pandas as pd
import numpy as np
import bz2

with bz2.open("cnn_dailymail_3.0.0_shorts.json.bz2", "rt", encoding="utf-8") as f:
    news_df = pd.read_json(f)

There are two labelled columns in the data set: `article` (the full news article), and `highlights` (highlights from the article, which we will treat as a “reference summary”).

The entire dataset contains 5,000 news articles, but since some of the techniques we will explore here can be quite compute-intensive (depending on the hardware used to run this lab), we limit ourselves to a selection of articles for some of the exercises:

- **If you are running this lab with a GPU** (strongly recommended), the selection contains 100 articles (ten hand-picked ones, plus 90 randomly sampled ones).
- **Without a GPU**, it only contains the ten hand-picked articles.

In [21]:
pd.set_option('display.max_colwidth', None)

import torch

indices = [6, 53, 56, 340, 730, 1940, 1983, 2404, 2826, 4673]
if torch.cuda.is_available():
    # With a GPU, add more (randomly sampled) articles, for a total of 100
    others = news_df.drop(index=indices).sample(90, random_state=42)
    indices += others.index.tolist()
short_news_df = news_df.loc[indices]
short_news_df

,article,highlights
6,"Former French President Nicolas Sarkozy was called in for questioning Tuesday as part of a corruption investigation in connection with an inquiry into alleged abuse of power. Financial prosecutor Eliane Houlette confirmed that Sarkozy was called in around 8 a.m. local time by police in Nanterre. Investigators have been looking into Sarkozy's campaign financing, CNN's French affiliate BFMTV reported. They are trying to establish whether the former President, who led the country from 2007 to 2012, obtained confidential information on legal cases concerning him from top magistrates in exchange for the offer of a prestigious post, the channel said. Under French law, Sarkozy can be held for questioning for a day, which can be extended once by another 24 hours. The situation is unprecedented for a former president, BFMTV reported. His lawyer, Thierry Herzog, and two high-ranking magistrates were called in for questioning in the Paris area Monday, according to BFMTV. Their police custody was extended by another 24 hours Tuesday. Sarkozy left office in May 2012 after he lost in the presidential election to Socialist rival Francois Hollande. In recent months, speculation has grown that he might be positioning himself for a political comeback at the head of his center-right party, the UMP. CNN's Jim Bittermann reported from Paris and Laura Smith-Spark wrote in London.","Former French President Nicolas Sarkozy is called in for questioning, prosecutor says . Investigators have been looking into Sarkozy's campaign financing . They want to establish if he obtained secret information about cases concerning him ."
53,"Thailand's King Bhumibol Adulyadej has been hospitalized with a high fever and irregular blood pressure, the palace said Saturday. The king, who was admitted to a hospital in Bangkok on Friday evening, appeared to be improving. ""On Saturday morning, his medical team announced that after being administered antibiotics intravenously, his blood pressure stabilized and his fever dropped,"" the palace said in a statement. The 86-year-old is a member of the Chakri dynasty, which has occupied the Thai throne since the 18th century. The U.S.-born and Swiss-educated king is a deeply revered figure whose unifying appeal stretches across all elements of Thai society, from the rich urban elite to poor provincial farmers. He ascended to the throne in 1946, and has reigned over more than 20 prime ministers, 17 military coups and 17 constitutions. CNN's Kocha Olarn reported from Bangkok, and Faith Karimi wrote and reported from Atlanta.",The king was admitted to a hospital in Bangkok . He has reigned over more than 20 prime ministers .
56,"A Disney Cruise Line employee has been reported missing during a seven-day cruise off the Mexico's Pacific coast, the company said Thursday. The unidentified worker on board the Disney Wonder failed to appear for a Tuesday shift, Disney spokeswoman Christi Erwin Donnan said. The vessel, which left Los Angeles on Sunday, docked Wednesday in Puerto Vallarta, Mexico, Donnan said. The U.S. Coast Guard, the Mexican navy and the Bahamas Maritime Authority have participated in the search and investigation, Donnan said. The Disney Wonder is registered in the Bahamas. The Mexican navy has been conducting searches since Tuesday, Disney Cruise Line said in a written statement. ""Given the circumstances, we are very concerned about this situation and are doing everything possible to assist with the search effort and investigation,"" it said. The company said it conducted a thorough inspection of the cruise ship. U.S. Coast Guard Petty Officer Pamela Manns said the Mexican navy is leading the search and asked for Coast Guard help early in the effort. The Coast Guard provided long-range search aircraft but is not now actively involved in the search. The FBI is not involved because it does not have jurisdiction, as the ship was off the coast of Mexico flying under a foreign flag, said spokeswoman La

## Problem 1: Extractive summarization

In this problem, we will produce a summary of a news article by extracting a small number of sentences from it, an approach known as _extractive_ summarization.

We will once again use [EmbeddingGemma2](https://developers.googleblog.com/embeddinggemma-2-the-developer-guide/) for this, the model you’ve already seen in previous labs.

In [22]:
from pathlib import Path
from liu_tm.torch import check_gpu

check_gpu()
# This part checks if we’re on the lab computers, and loads the model from there
model_name = Path("/courses/TDDE16/models/google/embeddinggemma-2")
if not model_name.exists():
    model_name = "google/embeddinggemma-2"

In [23]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    model_name,
    # Text only (270M parameters)
    config_kwargs={"vision_config": None, "audio_config": None},
)

Loading weights:   0%|          | 0/413 [00:00<?, ?it/s]

You should already know how to encode documents with this model, but let’s remind ourselves which different prompts there are to pick the best one for our use case:

In [24]:
model.prompts

{'query': 'task: search result | query: ',
 'document': 'title: none | text: ',
 'BitextMining': 'task: search result | query: ',
 'Classification': 'task: classification | query: ',
 'Clustering': 'task: clustering | query: ',
 'CodeRetrieval': 'task: code retrieval | query: ',
 'Document': 'title: none | text: ',
 'FactChecking': 'task: fact checking | query: ',
 'InstructionRetrieval': 'task: code retrieval | query: ',
 'MultilabelClassification': 'task: classification | query: ',
 'PairClassification': 'task: sentence similarity | query: ',
 'QuestionAnswering': 'task: question answering | query: ',
 'Reranking': 'task: search result | query: ',
 'Retrieval': 'task: search result | query: ',
 'Retrieval-document': 'title: none | text: ',
 'Retrieval-query': 'task: search result | query: ',
 'STS': 'task: sentence similarity | query: ',
 'SearchQuery': 'task: search result | query: ',
 'SentenceSimilarity': 'task: sentence similarity | query: ',
 'Summarization': 'task: sentence sim

You should already know how to encode documents and sentences with EmbeddingGemma2 by now. However, there is one important difference here: We don’t want to compute embeddings for entire documents, but for _each sentence_ in the document, so that we can compare their similarity later.  For this, you will also need to perform _sentence splitting_ before feeding the sentences into EmbeddingGemma2.  This can be done with the help of spaCy:

In [25]:
import spacy
nlp = spacy.load("en_core_web_md")

<div class="alert alert-info">

Complete the function below to **split a text into sentences** and return both the sentences and the **computed embeddings for them**, using the most appropriate prompt from EmbeddingGemma2.

(_Note:_ You probably want to turn the progress bar _off_ this time, since we are going to run this function many times in a row later.)

</div>

In [26]:
def get_sentences_and_embeddings(text):
    """Splits the text into sentences and computes embeddings for each of them.

    Arguments:
        text: The text to process, e.g. one entire news article.

    Returns:
        A tuple (sentences, sentence_vectors).  `sentences` should be a list of 
        sentences from the text, while `sentence_vectors` should be a list of
        embedding vectors corresponding to these sentences.
    """
    sentences = [sentence.text for sentence in nlp(text).sents]
    sentence_vectors = model.encode(
        sentences,
        prompt_name="Clustering",
        show_progress_bar=False,
    )
    return sentences, sentence_vectors

#### 🤞 Test your code

The following cell shows how your function should be called, and sanity-checks the returned value for one of the news articles:

In [27]:
sents, vecs = get_sentences_and_embeddings(short_news_df.iloc[2]["article"])

with check_solutions() as check:
    check.equals(len(sents), 13)  # should produce 13 sentences
    check.equals(len(vecs), 13)  # should return 13 vectors
    check.equals(sents[4], 'The Disney Wonder is registered in the Bahamas.')

### Implementing the maximum marginal relevance (MMR) algorithm

You now have all the necessary inputs to produce an extractive summary of a news article. Write a function that takes a list of sentences and their corresponding embedding vectors, as well as a number of sentences to extract from it. Your function should then implement the **maximum marginal relevance (MMR) algorithm** as follows:

1. Initially, your candidate set $C$ contains all sentences in the news article, and the set of selected sentences $S$ is empty.
2. As the “profile vector” $p$, use the **centroid** of all the sentence vectors from the news article.
3. Pick the next sentence to extract from $C\textbackslash S$ using the marginal relevance formula:

$$
s_i = \textrm{arg\,max}_{s \in C\textbackslash S} ~\left( \textrm{sim}(s, p) - \textrm{max}_{s_j \in S} ~\textrm{sim}(s, s_j) \right)
$$

In this formula, “sim” is the plain **cosine similarity** between the vectors.  You can compute it with `model.similarity()`, which you already know from Lab&nbsp;1.  Note that `model.similarity()` takes two lists of vectors and returns a _matrix_ with the cosine similarities between all pairs of vectors from the two lists.

4. Repeat step 3 until you have extracted $n$ sentences, where $n$ is given as an argument to your function.

In [28]:
def mmr_summarize(sentences, vectors, n=3):
    """Produce an extractive summary from a list of sentences and their vectors.

    Arguments:
        sentences (list): A list of sentences.
        vectors (list): A list of vectors, one for each sentence.
        n (int): The number of sentences to extract for the summary.

    Returns:
        A summary of `n` extracted sentences, as a single string.
    """
    if n <= 0 or not sentences:
        return ""

    vectors = np.asarray(vectors)
    profile_vector = vectors.mean(axis=0)
    candidates = list(range(len(sentences)))
    selected = []

    while candidates and len(selected) < n:
        candidate_vectors = vectors[candidates]
        relevance = model.similarity(candidate_vectors, [profile_vector]).squeeze(1)

        if selected:
            redundancy = model.similarity(
                candidate_vectors, vectors[selected]
            ).max(dim=1).values
            scores = relevance - redundancy
        else:
            scores = relevance

        next_sentence = candidates.pop(scores.argmax().item())
        selected.append(next_sentence)

    return " ".join(sentences[i] for i in selected)

#### 🤞 Test your code

The following cell puts it all together: it extracts the sentences and embeddings for one news article, generates the extractive summary, and displays it:

In [29]:
news = short_news_df.iloc[4]
sents, vecs = get_sentences_and_embeddings(news["article"])
summary = mmr_summarize(sents, vecs, n=3)

# Show the article, highlights, and extracted summary in a table
display(
    pd.DataFrame({
        "article": [news["article"]],
        "highlights": [news["highlights"]],
        "summary": [summary]
    })
)

,article,highlights,summary
0,"The Premier League has announced that next season's competition will begin on August 8. The Barclays Premier League season will be followed by the European Championship, and the Premier League's move could allow England manager Roy Hodgson additional time to prepare his side, should they qualify for the tournament. In a short Twitter posting, the Premier League stated: 'The 2015/16 BPL season will kick off on Saturday 8 August'. Next season's Premier League campaign will start on August 8 . The Premier League announced the news on Twitter on Wednesday . The move could allow Roy Hodgson more time to prepare his squad for Euro 2016, if England qualify . The season will finish early in the calendar due to the scheduling of the Euros and Champions League final. The FA Cup third round will be the first fixture to be played next year due to how the 2016 calendar falls resulting in less congestion during the holiday period. The date for the final day of the season has yet to be announced. It will not be the earliest start to a Premier League campaign, with that remaining the August 7 opening day of the 1999-2000 season.",The earliest the Premier League season has ever begun is August 7 . The next Premier League campaign will kick off on August 8 . The move could give England more time to prepare for Euro 2016 .,"Next season's Premier League campaign will start on August 8 . The move could allow Roy Hodgson more time to prepare his squad for Euro 2016, if England qualify . The FA Cup third round will be the first fixture to be played next year due to how the 2016 calendar falls resulting in less congestion during the holiday period."


If your implementation is correct, your summary should look like this:

> The Premier League has announced that next season's competition will begin on August 8. The move could allow Roy Hodgson more time to prepare his squad for Euro 2016, if England qualify . The season will finish early in the calendar due to the scheduling of the Euros and Champions League final.

Try it out with different news articles to see what happens!

## Problem 2: Evaluating text summarization with ROUGE-2

In this problem, you implement a ROUGE metric to automatically compare the extracted summaries against the “highlights” column of the data set.

Concretely, you should implement **ROUGE-2**, which is the version of the ROUGE metric based on bigram overlap between a system output and a reference.  In our case, the “system output” is the (concatenated) string of sentences from the generated summary, while the “reference” is the `highlights` column from the news data set.  The ROUGE-2 score is the F1-score computed from the **number of overlapping bigrams** compared against the **total number of bigrams** in the system output and the reference.

You will need to tokenize the inputs in order to compute the bigram overlap.  The caveat here is that different tokenizers may result in different ROUGE scores.  For the purpose of this problem, you should use spaCy to tokenize your input.  If in doubt, you can refer to [the spaCy 101 documentation on tokenization](https://spacy.io/usage/spacy-101#annotations-token).

<div class="alert alert-info">

Complete the function below to **implement the ROUGE-2 measure**!

</div>

In [30]:
def rouge_2(system, reference):
    """Compute the ROUGE-2 score between a system output and a reference.
    
    Arguments:
        system (str): The system output, as a single string.
        reference (str): The reference to compare against, as a single string.

    Returns:
        The F1-score of the ROUGE-2 metric between system output and reference.
    """
    from collections import Counter

    system_tokens = [token.text for token in nlp(system)]
    reference_tokens = [token.text for token in nlp(reference)]
    system_bigrams = Counter(zip(system_tokens, system_tokens[1:]))
    reference_bigrams = Counter(zip(reference_tokens, reference_tokens[1:]))

    system_count = sum(system_bigrams.values())
    reference_count = sum(reference_bigrams.values())

    overlap = sum((system_bigrams & reference_bigrams).values())
    precision = overlap / system_count
    recall = overlap / reference_count
    return 2 * precision * recall / (precision + recall) if overlap else 0.0

#### 🤞 Test your code

The following cell tests your implementation of ROUGE-2 with some toy examples:

In [31]:
with check_solutions() as check:
    check.equals(rouge_2("System output.", "Reference summary."), 0.0)  # Two strings without any bigram overlap
    check.equals(rouge_2("Two identical strings.", "Two identical strings."),  1.0)  # Two identical strings should return a score of one
    check.equals(rouge_2("This is my summary.", "This is the summary."), 0.5)  # When using spaCy’s tokenization, exactly half of the bigrams overlap, so the ROUGE-2 score should be 0.5

### Baselines for extractive summarization

Now that you have an implementation of an evaluation metric, you can use it to evaluate the extractive summaries on the selection of news articles in `short_news_df`.

<div class="alert alert-info">

**Generate summaries and ROUGE-2 scores** for all articles in `short_news_df`!

After running the following cell:
- `extractive` should contain the extractive summaries for all articles in `short_news_df`, one summary per article, with **two sentences per summary**.
- `extractive_rouge_2` should contain the average ROUGE-2 score for the extractive summaries, when evaluated against the "highlights" column in `short_news_df`.

</div>

In [32]:
extractive = []  # Summaries of all articles in `short_news_df`

for article in short_news_df["article"]:
    sentences, vectors = get_sentences_and_embeddings(article)
    extractive.append(mmr_summarize(sentences, vectors, n=2))

extractive_rouge_2 = np.mean([
    rouge_2(summary, highlights)
    for summary, highlights in zip(extractive, short_news_df["highlights"])
])

We cannot say much about the ROUGE-2 score unless we compare it against a **baseline**.  A simple baseline for text summarization is to just take the first $n$&nbsp;sentences from the article.  

<div class="alert alert-info">

**Compute the ROUGE-2 score of this baseline** in the cell below, i.e., the ROUGE-2 score when taking the **first two sentences** of the article as your “summary”.

</div>

In [33]:
baseline = [
    " ".join(sentence.text for sentence in list(nlp(article).sents)[:2])
    for article in short_news_df["article"]
]
baseline_rouge_2 = np.mean([
    rouge_2(summary, highlights)
    for summary, highlights in zip(baseline, short_news_df["highlights"])
])

#### 🤞 Test your code

The following cell prints both ROUGE-2 scores:

In [34]:
print(f"ROUGE-2 (baseline)  : {baseline_rouge_2:.4f}")
print(f"ROUGE-2 (extractive): {extractive_rouge_2:.4f}")

ROUGE-2 (baseline)  : 0.1450
ROUGE-2 (extractive): 0.1911


## Problem 3: Prompting an instruction-tuned LLM

In order to perform _abstractive_ summarization, we will turn to an open-weights, instruction-fine-tuned large language model (LLM).  Concretely, we will use [Gemma&nbsp;4&nbsp;12B](https://huggingface.co/google/gemma-4-12B-it), a model with 12&nbsp;billion parameters released by Google, which we load via the [Transformers](https://huggingface.co/docs/transformers/) library.  At its original 16-bit precision, this model would need around 24&nbsp;GB of GPU memory, which is too large even for the lab computers.  Therefore, we use a _quantized_ version of it. _Quantization_ is a technique to distill a model down to a smaller size, allowing it to run faster and on smaller hardware, while sacrificing some of its performance.  Here, we use the [bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes) library to store the model weights with only 4&nbsp;bits each, which brings the memory requirements down to around 8&nbsp;GB.

The model is loaded in the following cell. Please note:

- **If you are running this on the LiU computers,** the following cell will use an already quantized model that we stored in the shared course folder.
- **If you are running this on your own computer with an NVIDIA GPU (with at least 10&nbsp;GB of memory),** the following cell will download the original model from Hugging Face the first time you run it, which **requires approx.&nbsp;24&nbsp;GB of disk space!**  The model is then quantized while it is loaded, which can take a few minutes.
- **Otherwise,** the following cell will fall back to the smaller [Gemma&nbsp;4&nbsp;E2B](https://huggingface.co/google/gemma-4-E2B-it) model, which **requires approx.&nbsp;10&nbsp;GB of disk space** and also runs without a GPU, albeit very slowly.

In [35]:
from pathlib import Path
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from liu_tm.torch import check_gpu

check_gpu()

# Settings for quantizing a model to 4 bits while loading it (requires an NVIDIA GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# This part checks if we’re on the LiU computers, and loads the already quantized model from there
model_name = Path("/courses/TDDE16/models/gemma-4-12B-it-bnb-nf4")
if not model_name.exists():
    if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory >= 10 * 1024**3:
        model_name = "google/gemma-4-12B-it"
    else:
        # Fallback: a smaller model that also runs without a GPU
        model_name = Path("/Users/felixunterleiter/Documents/GitHub/TM-Labs/models/gemma-4-E2B-it")

quantization_config = bnb_config if torch.cuda.is_available() else None

print(f"Loading model: {model_name}")
processor = AutoProcessor.from_pretrained(model_name, local_files_only=True)
model = AutoModelForMultimodalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map="mps",
    local_files_only=True,
)

Loading model: /Users/felixunterleiter/Documents/GitHub/TM-Labs/models/gemma-4-E2B-it


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Generating text with this model happens in three steps:

1. The `processor` turns the input text into a sequence of _token IDs_.  (Gemma models expect every input to start with the special `<bos>` (_beginning of sequence_) token, so we add it manually here.)
2. The `model.generate()` function predicts a continuation of this sequence, token by token.
3. The `processor` turns the resulting token IDs back into text.

Let’s try it out:

In [36]:
inputs = processor(text="<bos>What is the capital of Sweden?", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=64)
display(outputs)
display(processor.decode(outputs[0]))

tensor([[     2,   3689,    563,    506,   5279,    529,  27556, 236881,    108,
            818,   5279,    529,  27556,    563,  54206, 236761,    106]],
       device='mps:0')

'<bos>What is the capital of Sweden?\n\nThe capital of Sweden is Stockholm.<turn|>'

**Run the cell above multiple times** until you get different output at least once!

You should notice two things:
1. The `generate()` function returns token IDs, not text, and these include the token IDs of our input; we need to decode them, and to extract the answer, we have to cut off the part that corresponds to our input.
2. The decoded text contains special markers such as `<turn|>` (and maybe others, like `<|channel>`) that we did not provide in the input — and depending on the run, the model might not even produce an answer at all.

Let’s look at the second issue first.  Instruction-tuned models like Gemma are fine-tuned with a specific _chat_ or _prompt template_ that structures the input in a pre-defined way; the markers in the output above are the model’s attempt to continue a conversation in this format.  There are different, clearly separated parts in the template, corresponding to different _roles_ of the respective input:

- The **system** role contains global instructions for the LLM (e.g. how to behave, or meta information such as its knowledge cutoff date).
- The **user** role contains instructions or questions from the user.
- The **assistant** role contains the responses of the LLM.

The processor provides a function for automatically applying the correct template for the loaded model; it takes a list of _messages_, which are text strings with their corresponding roles.  With `tokenize=False`, we can look at the result as a text string:

In [37]:
messages = [
    {
        "role": "system",
        "content": "Answer the user query as accurately as possible.",
    },
    {
        "role": "user",
        "content": "What is the capital of Sweden?",
    },
]
print(processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

<bos><|turn>system
Answer the user query as accurately as possible.<turn|>
<|turn>user
What is the capital of Sweden?<turn|>
<|turn>model



Each message is wrapped in `<|turn>` and `<turn|>` markers, together with its role.  The option `add_generation_prompt=True` adds the beginning of a new turn for the model (which Gemma calls `model` instead of `assistant`), so that the LLM will continue by generating a response.  For some models, the template also adds an empty “thought” section (`<|channel>thought` … `<channel|>`), which tells the model to answer directly instead of “thinking” about its answer first.

The following function puts all of this together: it applies the chat template to a list of messages, generates a response, and returns only the newly generated text, without any special tokens:

In [38]:
def generate(messages, max_new_tokens=256):
    """Generate the response of the LLM to a list of chat messages.

    Arguments:
        messages (list): A list of messages, each a dictionary with keys "role" and "content".
        max_new_tokens (int): The maximum number of tokens to generate.

    Returns:
        A text string containing the LLM’s response.
    """
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    # Cut off the input tokens, and decode only the response
    response = outputs[0, inputs["input_ids"].shape[-1]:]
    return processor.decode(response, skip_special_tokens=True).strip()

In [39]:
generate(messages)

'The capital of Sweden is **Stockholm**.'

Hopefully, the output now only contains the answer to the question, and no further markers!

### Understanding prompt templates

Your task now is to test the behaviour of prompt templates (and your understanding of them) with the following small exercise.

<div class="alert alert-info">

Complete the function below to **return the capital of a country** by prompting the LLM.  In particular, your function should:

- use the "system" message to make the LLM return only the desired answer, rather than a full sentence (i.e., not _"The capital of Sweden is \*\*Stockholm\*\*."_ but just _"Stockholm"_)
- use the "user" message to supply only the name of the country
- return only a text string with the generated answer

</div>

In [40]:
def get_capital_of(country):
    """Prompt an LLM for the capital of a country.
    
    Arguments:
        country (str): The name of the country.

    Returns:
        A text string containing the LLM’s response.
    """
    messages = [
        {"role": "system", "content": "Given this name of a country, respond with its capital city name, NOTHING else. Do not include any other text."},
        {"role": "user", "content": country},
    ]
    return generate(messages)

#### 🤞 Test your code

The following cell should return only “Stockholm”:

In [41]:
capital = get_capital_of("Sweden")
display(capital)

with check_solutions() as check:
    check.equals(capital, "Stockholm")

'Stockholm'

If your code works with the input “Sweden”, **re-run the function with different inputs** to see what happens.  Does the LLM return the correct answer even for lesser-known countries?  What happens if you give it input that’s not an actual country?

In [42]:
# Example — try out your own inputs!
get_capital_of("Iceland")

'Reykjavik'

## Problem 4: Abstractive summarization

We now have everything we need to perform abstractive summarization, by prompting our LLM to generate a summary based on the input text.

<div class="alert alert-info">

In the same vein as in Problem&nbsp;3, **write a function that returns a summary of the input text** by prompting the LLM.  Note the following:

- Since we could specify the desired number of sentences for the extractive summaries (in Problem&nbsp;1), we want to have the same functionality here, so construct your messages in a way that request the given number of sentences from the LLM.  (Note that this is not an absolute guarantee that the LLM will always follow this instruction, but it should do so in most cases.)
- As before, construct your messages in such a way that the returned response will only contain the requested summary, without any further text (such as “Sure, here is a summary...”).

</div>

In [43]:
def llm_summarize(text, n=3):
    """Produce an abstractive summary for a given text.
    
    Arguments:
        text (str): The text to summarize.
        n (int): The number of sentences the summary should contain.

    Returns:
        A text string containing the abstractive summary produced by the LLM.
    """
    messages = [
        {"role": "system", "content": f"Summarize the provided text in EXACTLY {n} sentences. Produce only this summary with no further additional information."},
        {"role": "user", "content": text},
    ]
    return generate(messages)

#### 🤞 Test your code

The following cell should produce a summary for one of the news articles:

In [45]:
news = short_news_df.iloc[4]
summary = llm_summarize(news["article"], n=3)

# Show the article, highlights, and extracted summary in a table
pd.DataFrame({
    "article": [news["article"]],
    "highlights": [news["highlights"]],
    "summary": [summary]
})

,article,highlights,summary
0,"The Premier League has announced that next season's competition will begin on August 8. The Barclays Premier League season will be followed by the European Championship, and the Premier League's move could allow England manager Roy Hodgson additional time to prepare his side, should they qualify for the tournament. In a short Twitter posting, the Premier League stated: 'The 2015/16 BPL season will kick off on Saturday 8 August'. Next season's Premier League campaign will start on August 8 . The Premier League announced the news on Twitter on Wednesday . The move could allow Roy Hodgson more time to prepare his squad for Euro 2016, if England qualify . The season will finish early in the calendar due to the scheduling of the Euros and Champions League final. The FA Cup third round will be the first fixture to be played next year due to how the 2016 calendar falls resulting in less congestion during the holiday period. The date for the final day of the season has yet to be announced. It will not be the earliest start to a Premier League campaign, with that remaining the August 7 opening day of the 1999-2000 season.",The earliest the Premier League season has ever begun is August 7 . The next Premier League campaign will kick off on August 8 . The move could give England more time to prepare for Euro 2016 .,The Premier League has announced that the next season's competition will begin on August 8th. This scheduling change might give England manager Roy Hodgson more time to prepare for Euro 2016 should they qualify. The season will conclude early due to the scheduling of the Euros and Champions League final.


### Generate all summaries & evaluate

The only thing left to do is to generate summaries for all the articles in our selection, compute their ROUGE-2 scores against the "highlights" column, and compare the results against both the baseline and the extractive summaries.

<div class="alert alert-info">

After running the following cell:
- `abstractive` should contain the abstractive summaries for all articles in `short_news_df`, one summary per article.
  -  **Important:** Since we used two sentences for the extractive summarization & baselines, we should also prompt for **two-sentence summaries** here.
- `abstractive_rouge_2` should contain the average ROUGE-2 score for the abstractive summaries, when evaluated against the "highlights" column.

</div>

_Note:_ This cell can take a while to run, depending on which model and hardware you are running it with.  You may want to [use a progress bar](https://rich.readthedocs.io/en/stable/progress.html) or print an update in each loop iteration to get an idea of how many articles have already been processed.

In [46]:
%%time

from tqdm.auto import tqdm

abstractive = []
for article in tqdm(short_news_df["article"], total=len(short_news_df), desc="Summarizing articles"):
    abstractive.append(llm_summarize(article, n=2))

abstractive_rouge_2 = np.mean([
    rouge_2(summary, highlights)
    for summary, highlights in zip(abstractive, short_news_df["highlights"])
])

Summarizing articles:   0%|          | 0/10 [00:00<?, ?it/s]

CPU times: user 8.09 s, sys: 698 ms, total: 8.78 s
Wall time: 17.3 s


#### 🤞 Test your code

The following cell prints all ROUGE-2 scores, and the cell after that displays a table with the highlights and generated summaries, side-by-side:

In [47]:
print(f"ROUGE-2 (baseline)   : {baseline_rouge_2:.4f}")
print(f"ROUGE-2 (extractive) : {extractive_rouge_2:.4f}")
print(f"ROUGE-2 (abstractive): {abstractive_rouge_2:.4f}")

ROUGE-2 (baseline)   : 0.1450
ROUGE-2 (extractive) : 0.1911
ROUGE-2 (abstractive): 0.1214


In [48]:
pd.set_option('display.max_colwidth', None)
pd.DataFrame({
    "highlights": short_news_df.loc[:, "highlights"],
    "extractive": extractive,
    "abstractive": abstractive
})

,highlights,extractive,abstractive
6,"Former French President Nicolas Sarkozy is called in for questioning, prosecutor says . Investigators have been looking into Sarkozy's campaign financing . They want to establish if he obtained secret information about cases concerning him .","Former French President Nicolas Sarkozy was called in for questioning Tuesday as part of a corruption investigation in connection with an inquiry into alleged abuse of power. In recent months, speculation has grown that he might be positioning himself for a political comeback at the head of his center-right party, the UMP.","Former French President Nicolas Sarkozy was summoned for questioning in connection with a corruption investigation focusing on alleged abuse of power and the improper use of confidential information. Investigators are examining whether Sarkozy traded information on legal cases for a prestigious position, marking an unprecedented situation for a former president."
53,The king was admitted to a hospital in Bangkok . He has reigned over more than 20 prime ministers .,"The king, who was admitted to a hospital in Bangkok on Friday evening, appeared to be improving. He ascended to the throne in 1946, and has reigned over more than 20 prime ministers, 17 military coups and 17 constitutions.","King Bhumibol Adulyadej was hospitalized with a high fever and irregular blood pressure but was reportedly improving after receiving intravenous antibiotics. This deeply revered 86-year-old monarch, who ascended to the throne in 1946, remains a unifying figure across all segments of Thai society."
56,A cruise line employee has not been seen since Tuesday . Disney Wonder left Los Angeles on Sunday . Mexican navy and Bahamas Maritime Authority are involved in search .,"The Mexican navy has been conducting searches since Tuesday, Disney Cruise Line said in a written statement. CNN's Dave Alsup and Sara Weisfeldt contributed to this report.","A Disney Cruise Line employee has been reported missing during a seven-day cruise off the Pacific coast of Mexico, prompting searches by the U.S. Coast Guard, the Mexican navy, and the Bahamas Maritime Authority. Disney is actively assisting with the investigation while noting that the FBI is not involved due to the ship's registration and location."
340,Iain Hesford made over 200 league appearances for Blackpool . He also played for Hull and Sunderland among other clubs . The former goalkeeper has passed away at the age of 54 .,"Former goalkeeper Iain Hesford, pictured playing for Hesford, has died at the age of 54 . Hesford made his debut for the Seasiders in 1977 before going on to play for Sunderland and Hull in a career which spanned three decades.","Blackpool has announced the passing of former goalkeeper Iain Hesford at the age of 54, who had a significant career including over 200 league appearances for the club. Hull City also expressed sadness over the death of Hesford, acknowledging his time playing for both clubs and Sunderland."
730,The earliest the Premier League season has ever begun is August 7 . The next Premier League campaign will kick off on August 8 . The move could give England more time to prepare for Euro 2016 .,"Next season's Premier League campaign will start on August 8 . The move could allow Roy Hodgson more time to prepare his squad for Euro 2016, if England qualify .","The Premier League has announced that next season's competition will begin on August 8th, followed by the European Championship. This scheduling change could provide England manager Roy Hodgson more preparation time if their team qualifies for the tournament."
1940,"The coffee triggered emergency codes, including one for a hijacking . The flight was en route from Chicago to Frankfurt, Germany . The passengers caught a flight to Germany a day later .","A United Airlines flight from Chicago to Frankfurt, Germany, was diverted to Toronto this week after the pilot dumped a cup of coffee on the plane's communication's equipm

As before, the summaries (and therefore the ROUGE scores) will most likely be different if you run the same prompt again, so we encourage you to run it at least twice before you write up your reflection!

## Individual reflection

<div class="alert alert-warning">

**After you have solved the lab,** save the notebook and **work on the following part individually.**

</div>

<div class="alert alert-info">

**Write a _brief_ reflection on the questions below.**

- You may discuss the questions with your lab partner, but remember to write them up individually.  **Do not copy** answers from your lab partner.
- You may add more cells below (even code cells!) if you wish.

</div>

1. How do you interpret the ROUGE-2 scores you got at the end of Problem&nbsp;4?  Do they agree with your own intuition of the quality of the summaries when looking at the outputs?  What conclusions do you draw from this experiment?  Discuss this with at least one concrete example from the final evaluation.
2. We have worked on several other tasks in Labs 1–4.  Could we have used an LLM like in this lab for these tasks as well?  What would be some of the advantages and disadvantages of doing that?

# A1
Based on the 10 articles, the extractive summary obtained by MMR yield the highest ROUGE-2 score, meaning that it had on average the most two-word overlap in those examples. Even though our abstractive summaries scored lowest, we cannot simply conclude that using an LLM for this task is least informative as ROUGE-2 simply focuses on matching wording between summarization and ground-truth but an LLM could express the same information with vastly different wording, whereas with MMR we generate summaries that will never use a different wording than was used in original text.
Based on the 10 samples it seems the abstractive summaries capture the same essence then the highlights, so it might be beneficial to consult different, more informative metrics, other than ROUGE-2, to judge more nuanced differences in summaries.

In [61]:
i = 4

print("Highlighs:")
print(short_news_df.iloc[i]["highlights"])
print("")
print("MMR summary:")
print(extractive[i])
print("")
print("LLM summary:")
print(abstractive[i])

Highlighs:
The earliest the Premier League season has ever begun is August 7 . The next Premier League campaign will kick off on August 8 . The move could give England more time to prepare for Euro 2016 .

MMR summary:
Next season's Premier League campaign will start on August 8 . The move could allow Roy Hodgson more time to prepare his squad for Euro 2016, if England qualify .

LLM summary:
The Premier League has announced that next season's competition will begin on August 8th, followed by the European Championship. This scheduling change could provide England manager Roy Hodgson more preparation time if their team qualifies for the tournament.


Article 4 is a great example for that, here the abstractive summary mentions the August 8 start and England’s extra prep time (so does the highlight) but phrases them slightly different. So its lower ROUGE-2 score therefore does not mean it's a worse summary.

# A2

Sure, I would not see why an LLM cannot be used to classify texts, link entities, or extract keywords for topic modelling. A good advantage is that we would not need to train a new model but a set of instructions and a few examples might be sufficient. This advantage can however be a curse as well: Since we have not trained to model it could very well misunderstand categories specific to our task/dataset. This can be especially problematic for nieche, domain-specific knowledge where the pretrained model was not sufficiently trained on. Also without use of pydantic models or similar, we need to have some guardrails to make sure the output of the model is consistent with the requested format and does not derail. It generally needs to be evaluated, if the increased cost of running a pretrained LLM outweighs training a small classifier or other embedding model.


**Congratulations on finishing this lab! 👍**

<div class="alert alert-info">

➡️ Remember that each student should submit their own version of this notebook, with their own individual answers to the reflection questions.

</div>